# 🐄 Boeuf Tracker — Colab (GPU CUDA + UI web)

Lance **toute l'app** sur le GPU Colab et donne une **URL publique** pour l'UI.
Ton code bascule tout seul en CUDA (pas de MLX sur Colab).

**Tu feras 2 choses à la main :**
1. glisser `behavior_video.pt` dans le panneau *Fichiers* (à gauche), ou l'uploader
2. uploader une vidéo à analyser

⚠️ Runtime → *Modifier le type d'exécution* → **GPU** avant de lancer.


## 1. Config — ton repo + ta branche


In [ ]:
GIT_URL    = 'https://github.com/ismaelgansonre/boeuf-tracker.git'  # <<< ton repo
GIT_BRANCH = 'desktop'                                             # <<< branche avec tes changements
GIT_TOKEN  = ''   # PAT GitHub si repo privé (sinon laisse vide)
PORT       = 8000

## 2. Installer + cloner le repo


In [ ]:
import subprocess, sys, os
from pathlib import Path

print('⏳ ffmpeg + dépendances...')
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'ultralytics>=8.0.0','transformers>=4.35.0','timm>=1.0.0',
    'opencv-python>=4.8.0','flask>=3.0.0','numba>=0.60.0','scikit-image>=0.22'], check=True)

url = GIT_URL
if GIT_TOKEN:
    url = url.replace('https://', f'https://{GIT_TOKEN}@')
REPO = '/content/' + GIT_URL.rstrip('/').split('/')[-1].replace('.git','')
if not Path(REPO).exists():
    subprocess.run(['git','clone','--depth','1','-b',GIT_BRANCH,url,REPO], check=True)
print('✅ repo :', REPO)

## 3. Vérifier le GPU


In [ ]:
import torch
assert torch.cuda.is_available(), '❌ Active le GPU : Runtime → type d\'exécution → GPU'
print('✅ GPU :', torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## 4. Placer `behavior_video.pt` (drag-and-drop ou upload)

Glisse le fichier dans le panneau *Fichiers* à gauche (dans `/content`),
puis exécute cette cellule. Sinon elle ouvre un sélecteur.


In [ ]:
import shutil, os
dst = os.path.join(REPO, 'behavior_video.pt')
if os.path.exists(dst):
    print('✅ déjà en place')
else:
    src = next((p for p in ['/content/behavior_video.pt',
                            '/content/drive/MyDrive/behavior_video.pt'] if os.path.exists(p)), None)
    if src:
        shutil.copy(src, dst); print('✅ copié depuis', src)
    else:
        from google.colab import files
        print('Glisse/uploade behavior_video.pt :')
        up = files.upload()
        for n in up:
            if n.endswith('.pt'): shutil.move(n, dst)
        print('✅ en place' if os.path.exists(dst) else '❌ échec')

# (optionnel) modèle absent = comportement vidéo désactivé, le reste tourne.
print('comportement vidéo :', 'ON' if os.path.exists(dst) else 'OFF (pas de .pt)')

## 5. Uploader une vidéo à analyser


In [ ]:
from google.colab import files
import shutil, os
os.makedirs('/content/videos', exist_ok=True)
print('Uploade une vidéo (mp4/mov) :')
up = files.upload()
VIDEO = None
for n in up:
    VIDEO = f'/content/videos/{n}'; shutil.move(n, VIDEO)
assert VIDEO, '❌ aucune vidéo uploadée'
print('✅ vidéo :', VIDEO)

## 6. Lancer l'app + URL publique

Tunnel Cloudflare (gratuit, **sans compte**). L'URL `*.trycloudflare.com`
s'affiche : clique dessus pour ouvrir l'UI.


In [ ]:
import subprocess, re, time, os, urllib.request

# cloudflared (sans compte)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && '
                   'chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

# app.py en CUDA (PAS de --mlx) ; Flask sert l'UI sur PORT
app = subprocess.Popen(['python','app.py','--source',VIDEO,
                        '--host','0.0.0.0','--port',str(PORT),'--device','auto'],
                       cwd=REPO)

print('⏳ démarrage du worker...')
for _ in range(90):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/', timeout=2); break
    except Exception: time.sleep(1)
else:
    print('⚠️ worker lent — le tunnel démarre quand même')

print('⏳ ouverture du tunnel...')
tun = subprocess.Popen(['cloudflared','tunnel','--url',f'http://localhost:{PORT}'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        print('\n🌐  UI EN LIGNE :', m.group(0), '\n')
        break

## 🎉 C'est en ligne

Ouvre l'URL `*.trycloudflare.com` affichée ci-dessus. Laisse cette cellule
tourner (elle maintient l'app + le tunnel). Pour changer de vidéo : refais
les cellules 5 et 6.
